In [0]:
%run ./env

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

'current_catalog : mal_maite_bi_preprd'

'current_schema : process_time_analyses'

'common_schema : common'

'current_location : abfss://maite_bi@adlsdpcompreprddata.dfs.core.windows.net/process_time_analyses/'

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./translation_function

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

'current_catalog : mal_maite_bi_preprd'

... (3 outputs left out)

# translation_function

Fonctions de construction des tables de traduction. Ce notebook ne produit
**aucune** table : il ne définit que des fonctions, appelées par
`build_translations` et `dim_trad_language`.

Même rôle et même emplacement que `delta_function` : un notebook de fonctions,
chargé par `%run`, partagé entre les notebooks d'alimentation.

## `build_translation_dim`

Rend dense une table `*_translations` source. Voir `build_translations` pour le
détail du raisonnement ; en résumé, trois problèmes sont traités ici :

1. le **fallback** du PO (langue → anglais → clé) n'est pas réalisable en RLS,
   qui sait supprimer des lignes mais pas en substituer une ;
2. les traductions sont **incomplètes** : une clé non traduite disparaîtrait du
   visuel après filtrage RLS, et les totaux seraient faux ;
3. le couple (clé, langue) doit être **unique** pour qu'il reste exactement une
   ligne par clé après filtrage.

## `trs_translations_forme_longue`

Les traductions des mesures TRS sont livrees au **format large** : une ligne par
cle, une colonne par langue (`label_fr`, `label_en`, `label_ro`, `label_cs`).
`build_translation_dim` attend l'inverse, une ligne par (cle, langue).

Cette fonction fait la conversion, et rien d'autre : une fois la source depliee,
le dedoublonnage, la grille dense et la chaine de repli sont ceux de toutes les
autres tables de traduction du projet.

> Format retenu par l'equipe `trs_metadata`. Le document de contrat en assume la
> contrepartie : ajouter une langue demande une migration de schema chez eux.
> Cote notre chaine, les langues ne sont pas codees en dur (voir ci-dessous).


## `publish_dim`

Écriture avec la mécanique Delta du projet, en `mode="full"` imposé.

En mode `update`, `handle_table_update` ne fait qu'insérer et mettre à jour : il
ne supprime jamais. Une clé retirée à la source resterait donc indéfiniment dans
la table du modèle et continuerait d'apparaître dans les slicers du rapport — un
libellé fantôme, sans données derrière. Ces tables sont petites et entièrement
redérivées à chaque run : le remplacement complet est plus sûr et sans coût
notable.

In [0]:
%run ./dim_trad_language

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

'current_catalog : mal_maite_bi_preprd'

... (15 outputs left out)

mal_maite_bi_preprd.common.dim_trad_language


_1
language
code
language_label
culture_code


['code', 'language_label', 'culture_code']


Table mal_maite_bi_preprd.common.dim_trad_language has been fully replaced (delete + insert).
Number of rows inserted: 4


In [0]:
%run ./build_nomenclatures

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

'current_catalog : mal_maite_bi_preprd'

... (55 outputs left out)

['code', 'sequence', 'production_workshop', 'process', 'deleted']
Table mal_maite_bi_preprd.common.dim_processes_phases has been fully replaced (delete + insert).
Number of rows inserted: 9


## Les 3 nomenclatures TRS

`trs_nomenclature` porte les cles canoniques du suivi des temps, reparties en
trois familles par `entity_type` : `measure`, `category`, `measure_type`.

On en tire trois dimensions, une par famille. Elles ne portent que la cle :
leur role n'est pas de porter des attributs mais **l'unicite**. Sans elles, la
table de traduction (plusieurs lignes par cle) et la table de faits (plusieurs
lignes par cle) se brancheraient directement l'une sur l'autre, ce que Power BI
ne peut traiter qu'en relation plusieurs-a-plusieurs.

Les cles correspondent aux valeurs de `trs_metadata` : `measure_DE` pour les
mesures, `categorie` et `measure_type` pour les deux autres.


mal_maite_bi_preprd.common.dim_trs_measure
Table mal_maite_bi_preprd.common.dim_trs_measure has been fully replaced (delete + insert).
Number of rows inserted: 74
mal_maite_bi_preprd.common.dim_trs_category
Table mal_maite_bi_preprd.common.dim_trs_category has been fully replaced (delete + insert).
Number of rows inserted: 10
mal_maite_bi_preprd.common.dim_trs_measure_type
Table mal_maite_bi_preprd.common.dim_trs_measure_type has been fully replaced (delete + insert).
Number of rows inserted: 5


## Périmètre couvert

**14 tables** dans `common`. Chaque table de traduction a désormais son côté
« 1 » : plus aucune relation plusieurs-à-plusieurs dans le modèle.

> **À ne pas oublier côté Power BI** : ces tables contiennent les lignes
> supprimées. Filtrer `deleted = false` à l'import, sinon les libellés obsolètes
> réapparaîtront dans les slicers.

In [0]:
%run ./build_translations

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

'current_catalog : mal_maite_bi_preprd'

... (57 outputs left out)

Clés sans aucune traduction, par table :


table,count
dim_trad_production_line_variable,2550
dim_trad_variable,327
dim_trad_localization_group,4


Détail, à transmettre à l'équipe front :


table,code_affiche
dim_trad_localization_group,germination
dim_trad_localization_group,kilning
dim_trad_localization_group,pre-germination
dim_trad_localization_group,steeping
dim_trad_production_line_variable,1
dim_trad_production_line_variable,10
dim_trad_production_line_variable,100
dim_trad_production_line_variable,101
dim_trad_production_line_variable,10152
dim_trad_production_line_variable,10158
